In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

path = "C:/projects/Final Rat Trans Project Result2023/blindmolerat_rnaseqanalysis-2024/DMBA/Results_10092025_f2.5_p0.01_SELECTED/genesymbol/"
os.chdir(path)

In [ ]:
def get_antilog(df_fold_change):
    df_fold_change.rename(columns={df_fold_change.columns[0]: "GeneSymbol", 
                                   df_fold_change.columns[1]: "GeneID"}, inplace=True)
    log2_fold_changes = df_fold_change['log2FoldChange']
    antilog_fold_changes = np.power(2, log2_fold_changes)
    df_fold_change['Antilog_FC'] = antilog_fold_changes   
    df_fold_change = df_fold_change.rename(columns = {'log2FoldChange': 'logFC'})
    df_fold_change = df_fold_change[['GeneSymbol', 'GeneID', 'logFC', 'Antilog_FC', 'padj']]  # 'logCPM', 
    return df_fold_change

In [ ]:
spalax_wound_foldchange = pd.read_csv("S.control-S.wound/S.control-S.wound_genes.csv") 
#spalax_wound_foldchange = spalax_wound_foldchange.rename(columns = {'Unnamed: 0': 'GeneID'})
spalax_wound_foldchange = get_antilog(spalax_wound_foldchange)

mouse_wound_foldchange = pd.read_csv("MM.control-M.wound/MM.control-M.wound_genes.csv") 
#mouse_wound_foldchange = mouse_wound_foldchange.rename(columns = {'Unnamed: 0': 'GeneID'})
mouse_wound_foldchange = get_antilog(mouse_wound_foldchange)

spalax_scar_foldchange = pd.read_csv("S.control-S.scar/S.control-S.scar_genes.csv") 
#spalax_scar_foldchange = spalax_scar_foldchange.rename(columns = {'Unnamed: 0': 'GeneID'})
spalax_scar_foldchange = get_antilog(spalax_scar_foldchange)

mouse_scar_foldchange = pd.read_csv("MM.control-M.scar/MM.control-M.scar_genes.csv") 
#mouse_scar_foldchange = mouse_scar_foldchange.rename(columns = {'Unnamed: 0': 'GeneID'})
mouse_scar_foldchange = get_antilog(mouse_scar_foldchange)

In [ ]:
spalax_wound_up_count = len(spalax_wound_foldchange[spalax_wound_foldchange['logFC'] > 0])
spalax_wound_down_count = len(spalax_wound_foldchange[spalax_wound_foldchange['logFC'] < 0])
spalax_scar_up_count = len(spalax_scar_foldchange[spalax_scar_foldchange['logFC'] > 0])
spalax_scar_down_count = len(spalax_scar_foldchange[spalax_scar_foldchange['logFC'] < 0])

mouse_wound_up_count = len(mouse_wound_foldchange[mouse_wound_foldchange['logFC'] > 0])
mouse_wound_down_count = len(mouse_wound_foldchange[mouse_wound_foldchange['logFC'] < 0])
mouse_scar_up_count = len(mouse_scar_foldchange[mouse_scar_foldchange['logFC']> 0])
mouse_scar_down_count = len(mouse_scar_foldchange[mouse_scar_foldchange['logFC'] < 0])

In [ ]:
spalax_wound_up_count + spalax_wound_down_count
spalax_scar_up_count + spalax_scar_down_count
mouse_wound_up_count + mouse_wound_down_count
mouse_scar_up_count + mouse_scar_down_count

In [ ]:
# create data
x = ['Spalax wound', 'Mouse wound', 'Spalax scar', 'Mouse scar']
y1 = np.array([spalax_wound_up_count, mouse_wound_up_count, spalax_scar_up_count, mouse_scar_up_count ])
y2 = np.array([spalax_wound_down_count, mouse_wound_down_count, spalax_scar_down_count, mouse_scar_down_count])

# plot bars in stack manner
plt.bar(x, y1, color='r')
plt.bar(x, y2, bottom=y1, color='g')
plt.xlabel("Time points", fontsize = 20)
plt.ylabel("Number of DEGs", fontsize = 20)
plt.legend(["Up-regulated genes", "Down-regulated genes"], fontsize = 8)
#plt.title("up/down-regulated DEGs in both species", fontsize = 20)
plt.xticks(fontsize=12)
plt.yticks(fontsize=15)
plt.savefig('upregulated_downregulated.jpg', format='jpeg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

colors = {
    'Spalax wound': '#1f77b4',  # blue
    'Spalax scab': '#2ca02c',   # green
    'Mouse wound': '#ff7f0e',   # orange
    'Mouse scab': '#d62728'     # red
}

# Build one dataframe for all 4 groups
violin_df = pd.DataFrame({
    'logFC': pd.concat([
        spalax_wound_foldchange['logFC'],        
        mouse_wound_foldchange['logFC'],
        spalax_scar_foldchange['logFC'],
        mouse_scar_foldchange['logFC']
    ], ignore_index=True),
    'Group': ['Spalax wound'] * len(spalax_wound_foldchange)
           + ['Mouse wound'] * len(mouse_wound_foldchange)
           + ['Spalax scab'] * len(spalax_scar_foldchange)
           + ['Mouse scab'] * len(mouse_scar_foldchange)
})

# Plot
plt.figure(figsize=(6, 4))
sns.violinplot(data=violin_df, x='Group', y='logFC', palette=colors)
plt.axhline(0, color='gray', linestyle='--')
# plt.title('Distribution of log₂ fold change across groups')
plt.ylabel('log₂ fold change')
plt.tight_layout()
plt.savefig('violin_logFC_groups.jpg', dpi=300)
plt.show()

# Plot
plt.figure(figsize=(6, 4))
sns.boxplot(data=violin_df, x='Group', y='logFC', palette=colors)
plt.axhline(0, color='gray', linestyle='--')
# plt.title('Distribution of log₂ fold change across groups')
plt.ylabel('log₂ fold change')
plt.tight_layout()
plt.savefig('box_logFC_groups.jpg', dpi=300)
plt.show()


# KEGG

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_pathway_intensity(ax, timepoint, title):
    pathways = timepoint.iloc[0:20][['Description', 'qvalue', 'Count']]
    pathways['Description'] = [p.split(' - Mus musculus (house mouse)')[0] for p in pathways['Description'].values]
    pathways['qvalue'] = -np.log(pathways['qvalue'])

    n_display = len(pathways)
    y_pos = np.arange(n_display)[::-1] + (20 - n_display)  # Shift bars to top of 10-slot space

    ax.barh(y_pos, pathways.qvalue, align='center', color='black', height=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pathways.Description, fontsize=15)
    ax.set_title(title, fontsize=15)
    ax.set_xlim(0, 40)
    ax.set_ylim(-0.5, 20.5)  # Reserve space for 10
    ax.set_xlabel('-log(q-value)', fontsize=20)
    ax.tick_params(axis='x', labelsize=15)




# Create a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(20, 12))

# Plot each pathway intensity on its respective subplot
plot_pathway_intensity(axs[0, 0], spalax_wound_pathways, 'Spalax Wound Pathways')
plot_pathway_intensity(axs[0, 1], spalax_scab_pathways, 'Spalax Scab Pathways')
plot_pathway_intensity(axs[1, 0], mouse_wound_pathways, 'Mouse Wound Pathways')
plot_pathway_intensity(axs[1, 1], mouse_scab_pathways, 'Mouse Scab Pathways')

# Adjust layout for better spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Increase the space between rows

# Show the plot
plt.show()


# Upregulated

In [ ]:
def plot_pathway_intensity(ax, timepoint, title):
    pathways = timepoint.iloc[0:20][['Description', 'qvalue', 'Count']]
    pathways['Description'] = [p.split(' - Mus musculus (house mouse)')[0] for p in pathways['Description'].values]
    pathways['qvalue'] = -np.log(pathways['qvalue'])

    n_display = len(pathways)
    y_pos = np.arange(n_display)[::-1] + (20 - n_display)  # Shift bars to top of 10-slot space

    ax.barh(y_pos, pathways.qvalue, align='center', color='black', height=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pathways.Description, fontsize=15)
    ax.set_title(title, fontsize=15)
    ax.set_xlim(0, 40)
    ax.set_ylim(-0.5, 20.5)  # Reserve space for 10
    ax.set_xlabel('-log(q-value)', fontsize=20)
    ax.tick_params(axis='x', labelsize=15)
    
spalax_wound_pathways = pd.read_csv("../entrezID/S.control-S.wound/Kegg_pathway.csvUp.csv")
spalax_wound_pathways = spalax_wound_pathways.drop(columns = ['Unnamed: 0'])
spalax_wound_pathways = filter_by_qvalue(spalax_wound_pathways)
spalax_scab_pathways = pd.read_csv("../entrezID/S.control-S.scar/Kegg_pathway.csvUp.csv")
spalax_scab_pathways = spalax_scab_pathways.drop(columns = ['Unnamed: 0'])
spalax_scab_pathways = filter_by_qvalue(spalax_scab_pathways)

spalax_wound_pathways_scab = filter_scab_q_value(spalax_scab_pathways, spalax_wound_pathways)
spalax_wound_pathways_scab = reindex_df(spalax_wound_pathways[["Description", "qvalue"]].head(20), spalax_wound_pathways_scab)
print("\n")
print(spalax_wound_pathways[["Description", "qvalue"]].head(10))
print(spalax_wound_pathways_scab[["Description", "qvalue"]].head(10))

# ---------------------------------
mouse_wound_pathways = pd.read_csv("../entrezID/MM.control-M.wound/Kegg_pathway.csvUp.csv")
mouse_wound_pathways = mouse_wound_pathways.drop(columns = ['Unnamed: 0'])
mouse_wound_pathways = filter_by_qvalue(mouse_wound_pathways)
mouse_scab_pathways = pd.read_csv("../entrezID/MM.control-M.scar/Kegg_pathway.csvUp.csv")
mouse_scab_pathways = mouse_scab_pathways.drop(columns = ['Unnamed: 0'])
mouse_scab_pathways = filter_by_qvalue(mouse_scab_pathways)

mouse_wound_pathways_scab = filter_scab_q_value(mouse_scab_pathways, mouse_wound_pathways)
mouse_wound_pathways_scab = reindex_df(mouse_wound_pathways[["Description", "qvalue"]].head(20), mouse_wound_pathways_scab)
print("\n")
print(mouse_wound_pathways[["Description", "qvalue"]].head(10))
print(mouse_wound_pathways_scab.head(10))

# Create a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(20, 12))

# Plot each pathway intensity on its respective subplot
plot_pathway_intensity(axs[0, 0], spalax_wound_pathways, 'Spalax Wound Pathways')
plot_pathway_intensity(axs[0, 1], spalax_scab_pathways, 'Spalax Scab Pathways')
plot_pathway_intensity(axs[1, 0], mouse_wound_pathways, 'Mouse Wound Pathways')
plot_pathway_intensity(axs[1, 1], mouse_scab_pathways, 'Mouse Scab Pathways')

# Adjust layout for better spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Increase the space between rows

# Show the plot
plt.show()

# Downregulated

In [ ]:
def plot_pathway_intensity(ax, timepoint, title):
    pathways = timepoint.iloc[0:20][['Description', 'qvalue', 'Count']]
    pathways['Description'] = [p.split(' - Mus musculus (house mouse)')[0] for p in pathways['Description'].values]
    pathways['qvalue'] = -np.log(pathways['qvalue'])

    n_display = len(pathways)
    y_pos = np.arange(n_display)[::-1] + (20 - n_display)  # Shift bars to top of 10-slot space

    ax.barh(y_pos, pathways.qvalue, align='center', color='black', height=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pathways.Description, fontsize=15)
    ax.set_title(title, fontsize=15)
    ax.set_xlim(0, 45)
    ax.set_ylim(-0.5, 20.5)  # Reserve space for 10
    ax.set_xlabel('-log(q-value)', fontsize=20)
    ax.tick_params(axis='x', labelsize=15)
    
spalax_wound_pathways = pd.read_csv("../entrezID/S.control-S.wound/Kegg_pathway.csvDown.csv")
spalax_wound_pathways = spalax_wound_pathways.drop(columns = ['Unnamed: 0'])
spalax_wound_pathways = filter_by_qvalue(spalax_wound_pathways)
spalax_scab_pathways = pd.read_csv("../entrezID/S.control-S.scar/Kegg_pathway.csvDown.csv")
spalax_scab_pathways = spalax_scab_pathways.drop(columns = ['Unnamed: 0'])
spalax_scab_pathways = filter_by_qvalue(spalax_scab_pathways)

spalax_wound_pathways_scab = filter_scab_q_value(spalax_scab_pathways, spalax_wound_pathways)
spalax_wound_pathways_scab = reindex_df(spalax_wound_pathways[["Description", "qvalue"]].head(20), spalax_wound_pathways_scab)
print("\n")
print(spalax_wound_pathways[["Description", "qvalue"]].head(10))
print(spalax_wound_pathways_scab[["Description", "qvalue"]].head(10))

# ---------------------------------
mouse_wound_pathways = pd.read_csv("../entrezID/MM.control-M.wound/Kegg_pathway.csvDown.csv")
mouse_wound_pathways = mouse_wound_pathways.drop(columns = ['Unnamed: 0'])
mouse_wound_pathways = filter_by_qvalue(mouse_wound_pathways)
mouse_scab_pathways = pd.read_csv("../entrezID/MM.control-M.scar/Kegg_pathway.csvDown.csv")
mouse_scab_pathways = mouse_scab_pathways.drop(columns = ['Unnamed: 0'])
mouse_scab_pathways = filter_by_qvalue(mouse_scab_pathways)

mouse_wound_pathways_scab = filter_scab_q_value(mouse_scab_pathways, mouse_wound_pathways)
mouse_wound_pathways_scab = reindex_df(mouse_wound_pathways[["Description", "qvalue"]].head(20), mouse_wound_pathways_scab)
print("\n")
print(mouse_wound_pathways[["Description", "qvalue"]].head(10))
print(mouse_wound_pathways_scab.head(10))

# Create a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(20, 12))

# Plot each pathway intensity on its respective subplot
plot_pathway_intensity(axs[0, 0], spalax_wound_pathways, 'Spalax Wound Pathways')
plot_pathway_intensity(axs[0, 1], spalax_scab_pathways, 'Spalax Scab Pathways')
plot_pathway_intensity(axs[1, 0], mouse_wound_pathways, 'Mouse Wound Pathways')
plot_pathway_intensity(axs[1, 1], mouse_scab_pathways, 'Mouse Scab Pathways')

# Adjust layout for better spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Increase the space between rows

# Show the plot
plt.show()

# BP

In [ ]:
spalax_wound_bp = pd.read_csv("../entrezID/S.control-S.wound/GeneOntologyBP.csv")
spalax_wound_bp = spalax_wound_bp.drop(columns = ['Unnamed: 0'])
spalax_wound_bp = filter_by_qvalue(spalax_wound_bp)
spalax_scab_bp = pd.read_csv("../entrezID/S.control-S.scar/GeneOntologyBP.csv")
spalax_scab_bp = spalax_scab_bp.drop(columns = ['Unnamed: 0'])
spalax_scab_bp = filter_by_qvalue(spalax_scab_bp)

spalax_wound_bp_scab = filter_scab_q_value(spalax_scab_bp, spalax_wound_bp)
spalax_wound_bp_scab = reindex_df(spalax_wound_bp[["Description", "qvalue"]].head(10), spalax_wound_bp_scab)
print("\n")
print(spalax_wound_bp[["Description", "qvalue"]].head(10))
print(spalax_wound_bp_scab.head(10))


mouse_wound_bp = pd.read_csv("../entrezID/MM.control-M.wound/GeneOntologyBP.csv")
mouse_wound_bp = mouse_wound_bp.drop(columns = ['Unnamed: 0'])
mouse_wound_bp = filter_by_qvalue(mouse_wound_bp)
mouse_scab_bp = pd.read_csv("../entrezID/MM.control-M.scar/GeneOntologyBP.csv")
mouse_scab_bp = mouse_scab_bp.drop(columns = ['Unnamed: 0'])
mouse_scab_bp = filter_by_qvalue(mouse_scab_bp)

mouse_wound_bp_scab = filter_scab_q_value(mouse_scab_bp, mouse_wound_bp)
mouse_wound_bp_scab = reindex_df(mouse_wound_bp[["Description", "qvalue"]].head(20), mouse_wound_bp_scab)
print("\n")
print(mouse_wound_bp[["Description", "qvalue"]].head(10))
print(mouse_wound_bp_scab.head(10))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_pathway_intensity(ax, timepoint, title):
    pathways = timepoint.iloc[0:20][['Description', 'qvalue']]
    pathways['Description'] = [p.split(' - Mus musculus (house mouse)')[0] for p in pathways['Description'].values]
    pathways['qvalue'] = -np.log(pathways['qvalue'])

    y_pos = np.arange(len(pathways))[::-1]
    ax.barh(y_pos, pathways.qvalue, align='center', color='black')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pathways.Description, fontsize=15)  # Increase y tick label size
    ax.set_title(title, fontsize=15)
    ax.set_xlim(0, 55)
    ax.set_xlabel('-log(q-value)', fontsize=20)
    ax.tick_params(axis='x', labelsize=15)  # Increase x tick label size


# Create a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(20, 12))

# Plot each pathway intensity on its respective subplot
plot_pathway_intensity(axs[0, 0], spalax_wound_bp, 'Spalax Wound biological process')
plot_pathway_intensity(axs[0, 1], spalax_scab_bp, 'Spalax Scab biological process')
plot_pathway_intensity(axs[1, 0], mouse_wound_bp, 'Mouse Wound biological process')
plot_pathway_intensity(axs[1, 1], mouse_scab_bp, 'Mouse Scab biological process')

# Adjust layout for better spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Increase the space between rows

# Show the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(20, 12))

# Plot each pathway intensity on its respective subplot
plot_pathway_intensity(axs[0, 0], spalax_wound_bp, 'Spalax Wound biological process')
plot_pathway_intensity(axs[0, 1], spalax_wound_bp_scab, 'Spalax Wound biological process at spalax scab')
plot_pathway_intensity(axs[1, 0], mouse_wound_bp, 'Mouse Wound biological process')
plot_pathway_intensity(axs[1, 1], mouse_wound_bp_scab, 'Mouse wound biological process at mouse scab')

# Adjust layout for better spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Increase the space between rows

# Show the plot
plt.show()


# Upregulated

In [ ]:
spalax_wound_bp = pd.read_csv("../entrezID/S.control-S.wound/GeneOntologyBPUp.csv")
spalax_wound_bp = spalax_wound_bp.drop(columns = ['Unnamed: 0'])
spalax_wound_bp = filter_by_qvalue(spalax_wound_bp)
spalax_scab_bp = pd.read_csv("../entrezID/S.control-S.scar/GeneOntologyBPUp.csv")
spalax_scab_bp = spalax_scab_bp.drop(columns = ['Unnamed: 0'])
spalax_scab_bp = filter_by_qvalue(spalax_scab_bp)

spalax_wound_bp_scab = filter_scab_q_value(spalax_scab_bp, spalax_wound_bp)
spalax_wound_bp_scab = reindex_df(spalax_wound_bp[["Description", "qvalue"]].head(10), spalax_wound_bp_scab)
print("\n")
print(spalax_wound_bp[["Description", "qvalue"]].head(10))
print(spalax_wound_bp_scab.head(10))


mouse_wound_bp = pd.read_csv("../entrezID/MM.control-M.wound/GeneOntologyBPUp.csv")
mouse_wound_bp = mouse_wound_bp.drop(columns = ['Unnamed: 0'])
mouse_wound_bp = filter_by_qvalue(mouse_wound_bp)
mouse_scab_bp = pd.read_csv("../entrezID/MM.control-M.scar/GeneOntologyBPUp.csv")
mouse_scab_bp = mouse_scab_bp.drop(columns = ['Unnamed: 0'])
mouse_scab_bp = filter_by_qvalue(mouse_scab_bp)

mouse_wound_bp_scab = filter_scab_q_value(mouse_scab_bp, mouse_wound_bp)
mouse_wound_bp_scab = reindex_df(mouse_wound_bp[["Description", "qvalue"]].head(20), mouse_wound_bp_scab)
print("\n")
print(mouse_wound_bp[["Description", "qvalue"]].head(10))
print(mouse_wound_bp_scab.head(10))

# Create a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(20, 12))

# Plot each pathway intensity on its respective subplot
plot_pathway_intensity(axs[0, 0], spalax_wound_bp, 'Spalax Wound biological process')
plot_pathway_intensity(axs[0, 1], spalax_scab_bp, 'Spalax Scab biological process')
plot_pathway_intensity(axs[1, 0], mouse_wound_bp, 'Mouse Wound biological process')
plot_pathway_intensity(axs[1, 1], mouse_scab_bp, 'Mouse Scab biological process')

# Adjust layout for better spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Increase the space between rows

# Show the plot
plt.show()

# Downregulated

In [ ]:
def plot_pathway_intensity(ax, timepoint, title):
    pathways = timepoint.iloc[0:20][['Description', 'qvalue']]
    pathways['Description'] = [p.split(' - Mus musculus (house mouse)')[0] for p in pathways['Description'].values]
    pathways['qvalue'] = -np.log(pathways['qvalue'])

    y_pos = np.arange(len(pathways))[::-1]
    ax.barh(y_pos, pathways.qvalue, align='center', color='black')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pathways.Description, fontsize=15)  # Increase y tick label size
    ax.set_title(title, fontsize=15)
    ax.set_xlim(0, 65)
    ax.set_xlabel('-log(q-value)', fontsize=20)
    ax.tick_params(axis='x', labelsize=15)  # Increase x tick label size
    
spalax_wound_bp = pd.read_csv("../entrezID/S.control-S.wound/GeneOntologyBPDown.csv")
spalax_wound_bp = spalax_wound_bp.drop(columns = ['Unnamed: 0'])
spalax_wound_bp = filter_by_qvalue(spalax_wound_bp)
spalax_scab_bp = pd.read_csv("../entrezID/S.control-S.scar/GeneOntologyBPDown.csv")
spalax_scab_bp = spalax_scab_bp.drop(columns = ['Unnamed: 0'])
spalax_scab_bp = filter_by_qvalue(spalax_scab_bp)

spalax_wound_bp_scab = filter_scab_q_value(spalax_scab_bp, spalax_wound_bp)
spalax_wound_bp_scab = reindex_df(spalax_wound_bp[["Description", "qvalue"]].head(10), spalax_wound_bp_scab)
print("\n")
print(spalax_wound_bp[["Description", "qvalue"]].head(10))
print(spalax_wound_bp_scab.head(10))


mouse_wound_bp = pd.read_csv("../entrezID/MM.control-M.wound/GeneOntologyBPDown.csv")
mouse_wound_bp = mouse_wound_bp.drop(columns = ['Unnamed: 0'])
mouse_wound_bp = filter_by_qvalue(mouse_wound_bp)
mouse_scab_bp = pd.read_csv("../entrezID/MM.control-M.scar/GeneOntologyBPDown.csv")
mouse_scab_bp = mouse_scab_bp.drop(columns = ['Unnamed: 0'])
mouse_scab_bp = filter_by_qvalue(mouse_scab_bp)

mouse_wound_bp_scab = filter_scab_q_value(mouse_scab_bp, mouse_wound_bp)
mouse_wound_bp_scab = reindex_df(mouse_wound_bp[["Description", "qvalue"]].head(20), mouse_wound_bp_scab)
print("\n")
print(mouse_wound_bp[["Description", "qvalue"]].head(10))
print(mouse_wound_bp_scab.head(10))

# Create a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(20, 12))

# Plot each pathway intensity on its respective subplot
plot_pathway_intensity(axs[0, 0], spalax_wound_bp, 'Spalax Wound biological process')
plot_pathway_intensity(axs[0, 1], spalax_scab_bp, 'Spalax Scab biological process')
plot_pathway_intensity(axs[1, 0], mouse_wound_bp, 'Mouse Wound biological process')
plot_pathway_intensity(axs[1, 1], mouse_scab_bp, 'Mouse Scab biological process')

# Adjust layout for better spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)  # Increase the space between rows

# Show the plot
plt.show()